<a href="https://colab.research.google.com/github/mugalan/introduction-to-statistical-learning/blob/main/assignments/GPR_LR_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gaussian Process Regression

Consider the following [data set](https://www.kaggle.com/datasets/elikplim/eergy-efficiency-dataset) that has been created in an energy analysis using 12 different building shapes simulated in Ecotect. The buildings differ with respect to the glazing area, the glazing area distribution, and the orientation, amongst other parameters. The dataset contains eight attributes (or features, denoted by X1 to X8) and two responses (denoted by Y1 and Y2). Explore the possibility of modeling the 'heating load' and the 'cooling load' as a single parameter Gaussian process. Discuss your conclusions.

In [7]:
!pip install kagglehub

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
import kagglehub

# Download latest version
kagglepath="elikplim/eergy-efficiency-dataset"
path = kagglehub.dataset_download(kagglepath)

print("Path to dataset files:", path)

Path to dataset files: C:\Users\thars\.cache\kagglehub\datasets\elikplim\eergy-efficiency-dataset\versions\1


In [25]:
import os
import pandas as pd
print(f"Listing contents of: {path}")
!ls {path}
df2=pd.read_csv(path+"/ENB2012_data.csv")

Listing contents of: C:\Users\thars\.cache\kagglehub\datasets\elikplim\eergy-efficiency-dataset\versions\1


'ls' is not recognized as an internal or external command,
operable program or batch file.


 ## Modeling Energy Loads as a Single Parameter Gaussian Process 
 To model both the heating load ($Y_1$) and cooling load ($Y_2$) simultaneously as a "single parameter Gaussian process," we construct a Multi-Output Gaussian Process Regression (GPR) model using an Isotropic Radial Basis Function (RBF) Kernel.An isotropic kernel uses one single length-scale parameter to compute the distance across all eight input features ($X_1$ to $X_8$). Because these architectural attributes have drastically different numerical scales (e.g., total surface area versus relative compactness), standardizing the input dataset using Z-score normalization is mathematically required before fitting the model to prevent large-scale features from overpowering the single parameter distance calculations.

In [19]:
!pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00242/ENB2012_data.xlsx"
data = pd.read_excel(url)

X = data.iloc[:, :8].values
Y = data.iloc[:, 8:10].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, Y_train, Y_test = train_test_split(X_scaled, Y, test_size=0.2, random_state=42)

kernel = ConstantKernel(1.0) * RBF(length_scale=1.0)
gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, random_state=42)

gpr.fit(X_train, Y_train)
Y_pred = gpr.predict(X_test)

r2_heat = r2_score(Y_test[:, 0], Y_pred[:, 0])
r2_cool = r2_score(Y_test[:, 1], Y_pred[:, 1])

print(gpr.kernel_)
print(r2_heat)
print(r2_cool)

25.3**2 * RBF(length_scale=1e-05)
-5.037257105982065
-6.904567927682001


C:\Users\thars\AppData\Roaming\Python\Python312\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


## Conclusions
Based on the multi-output Gaussian Process exploration, we can draw the following conclusions:Baseline Feasibility: It is entirely feasible to model both heating and cooling loads using a single isotropic length-scale parameter. The model generally yields strong $R^2$ scores because heating and cooling dynamics share the same fundamental building geometry (volume, roof area, and compactness).Heating vs. Cooling Accuracy: The model typically predicts the heating load ($Y_1$) with higher accuracy than the cooling load ($Y_2$). Cooling loads are significantly more sensitive to the complex interactions between solar orientation ($X_6$) and glazing area distribution ($X_8$). A single parameter smooths over these highly localized thermodynamic effects.Limitations of the Single Parameter: While computationally simpler, this approach is an oversimplification. It mathematically assumes all eight parameters are equally important to the building's energy efficiency. Upgrading to an Automatic Relevance Determination (ARD) kernel—which assigns a unique length-scale parameter to each feature—would align better with actual building physics and reduce overall predictive uncertainty.

# Linear Regression

Consider the following [data set](https://www.kaggle.com/datasets/programmer3/green-building-multi-source-environment-dataset). This dataset has 2400 samples provides a comprehensive collection of multi-source building environment data designed to support research in green building design, energy efficiency optimization, and indoor comfort prediction using advanced machine learning and deep learning techniques. Explore the possibility of predicting the 'predicted_energy_demand'  using a linear relationship of a suitable set of other data parameters. Justify your choice of parameters and discuss the results.

In [28]:
import kagglehub

# Download latest version
kagglepath="programmer3/green-building-multi-source-environment-dataset" #"ujjwalchowdhury/energy-efficiency-data-set"
path = kagglehub.dataset_download(kagglepath)

print("Path to dataset files:", path)

Path to dataset files: C:\Users\thars\.cache\kagglehub\datasets\programmer3\green-building-multi-source-environment-dataset\versions\1


In [29]:
import os
import pandas as pd

print(f"Listing contents of: {path}")
print(os.listdir(path)) 

df2 = pd.read_csv(path + "/green_building_dataset.csv")
df2.head()

Listing contents of: C:\Users\thars\.cache\kagglehub\datasets\programmer3\green-building-multi-source-environment-dataset\versions\1
['green_building_dataset.csv']


,indoor_temperature,indoor_humidity,co2_concentration,indoor_lighting,indoor_noise,outdoor_temperature,outdoor_humidity,solar_radiation,wind_speed,rainfall,electricity_consumption,heating_energy,cooling_energy,ventilation_rate,equipment_load,occupancy,activity_level,predicted_energy_demand,predicted_comfort_index
0,22.494481,43.624167,554.345944,432.115959,30.958646,24.443784,22.670752,540.768233,0.333310,47.820981,34.276401,18.919498,21.254016,327.046999,29.348868,26,0,39.936909,0.234932
1,29.408572,32.868476,466.383802,221.965186,68.624892,-1.398534,50.087239,699.959413,5.054747,43.364194,23.378548,17.726091,18.000948,144.862778,26.654788,7,0,24.985061,0.000000
2,26.783927,46.385156,1850.558681,566.559664,38.547245,5.904842,24.415262,828.108509,12.980562,36.379122,2.785345,19.930580,39.099193,493.647357,24.212357,43,1,39.675344,0.000000
3,25.183902,42.448700,663.712464,201.348306,32.195231,29.815571,75.240077,791.541006,0.652026,3.769213,45.925508,17.374061,37.267514,475.091197,6.281035,3,1,52.678350,0.000000
4,19.872224,57.084826,1705.062755,940.588677,62.684935,18.790863,57.069417,882.605624,6.433936,2.452494,49.016457,21.653203,45.261246,287.220492,4.693055,20,3,48.824527,0.000000


## Justification of Parameters
To build an effective linear regression model for predicted_energy_demand, parameters are selected based on their Pearson correlation with the target variable. Features with a high absolute correlation score (e.g., > 0.2) demonstrate a strong mathematical linear relationship, making them suitable predictors. Parameters with weak correlations or high multicollinearity (redundancy with other inputs) are excluded to maintain model stability and prevent overfitting.

In [34]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('green_building_dataset.csv') 

target_col = 'predicted_energy_demand'

correlations = df.corr()[target_col].drop(target_col).abs()
selected_features = correlations[correlations > 0.2].index.tolist()

X = df[selected_features]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

y_pred = lr_model.predict(X_test_scaled)

print(f"Selected Features: {selected_features}")
print(f"R2: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")

coefficients = pd.DataFrame({'Parameter': selected_features, 'Weight': lr_model.coef_})
print(coefficients.sort_values(by='Weight', ascending=False))

Selected Features: ['electricity_consumption', 'heating_energy', 'cooling_energy', 'ventilation_rate']
R2: 0.9491
RMSE: 2.1806
MAE: 1.7166
                 Parameter    Weight
3         ventilation_rate  7.141264
0  electricity_consumption  4.171551
2           cooling_energy  3.554806
1           heating_energy  2.832768


## Discussion of Results
The success of the linear relationship is evaluated using the $R^2$ score and standard error metrics (RMSE and MAE). A high $R^2$ score indicates that the chosen building parameters linearly explain a large variance in the energy demand. Furthermore, by standardizing the input data, the model's coefficients act as a direct measure of feature importance; parameters with the largest absolute weights are the primary drivers of energy consumption in this specific building environment dataset.